# Фрагменти й копії в Python

У цьому ноутбуці розбираємо, коли Python створює нові обʼєкти, а коли лише додає нові посилання на старі дані. Головна ідея: копія не завжди означає повну незалежність.


## Поверхневе копіювання

`copy()` створює новий контейнер верхнього рівня. Але вкладені словники, списки та інші обʼєкти залишаються спільними.


In [ ]:
from copy import copy

mountains = {
    "Carpathians": {
        "Hoverla": 2061,
        "Pip_Ivan": 2022,
    },
    "Podillia": {
        "Tovtry": 401,
        "Medobory": 440,
    },
}

backup = copy(mountains)

print("Однаковий вміст:", backup == mountains)
print("Той самий обʼєкт:", backup is mountains)
print("Спільні Carpathians:", backup["Carpathians"] is mountains["Carpathians"])


На що звернути увагу: сам словник `backup` новий, але його вкладені частини поки що ті самі. Це швидко й економно, але не дає повної ізоляції.


In [ ]:
mountains["Crimea"] = {
    "Roman_Kosh": 1545,
    "Ai_Petri": 1234,
    "Eklizi_Burun": 1527,
}

print("Original keys:", mountains.keys())
print("Backup keys  :", backup.keys())
print("Однаковий вміст:", backup == mountains)


Додавання нового ключа у верхній рівень не потрапляє в `backup`. Це зміна самого контейнера, а не спільного вкладеного обʼєкта.


In [ ]:
mountains["Carpathians"]["Petros"] = 2020

print("Original Carpathians:", mountains["Carpathians"])
print("Backup Carpathians  :", backup["Carpathians"])
print("Спільний вкладений словник:", backup["Carpathians"] is mountains["Carpathians"])


Тут зміна вже видима в обох структурах. Ми змінили вкладений словник, а він спільний для оригіналу й поверхневої копії.


## Глибоке копіювання

`deepcopy()` проходить по вкладених обʼєктах і створює нові копії на кожному рівні. Це дорожче по памʼяті, але дає незалежність даних.
У минулому прикладі замініть copy на deepcopy та поспостерігайте за змінами.


In [ ]:
from copy import copy, deepcopy

base_request = {
    "endpoint": "/api/search",
    "params": {
        "query": "python",
        "limit": 10,
        "filters": ["articles", "videos"],
    },
    "headers": {
        "X-Trace-Id": "req-001",
    },
}

shallow_request = copy(base_request)
deep_request = deepcopy(base_request)

shallow_request["params"]["limit"] = 100
shallow_request["headers"]["X-Trace-Id"] = "req-002"

deep_request["params"]["filters"].append("books")
deep_request["headers"]["X-Trace-Id"] = "req-003"

print("Original:", base_request)
print("Shallow :", shallow_request)
print("Deep    :", deep_request)


Що тут відбувається: поверхнева копія змінює спільні `params` і `headers`, тому зачіпає оригінал. Глибока копія має власні вкладені структури, тому її зміни ізольовані.


## Рекурсивні структури

У дерев і графів часто є посилання назад: дитина знає свого батька. `deepcopy()` вміє зберігати такі звʼязки коректно, не створюючи нескінченну рекурсію.


In [ ]:
from copy import copy, deepcopy

category = {
    "name": "Python",
    "children": [],
}

subcategory = {
    "name": "Memory",
    "children": [],
    "parent": category,
}

category["children"].append(subcategory)

shallow_category = copy(category)
deep_category = deepcopy(category)

print("Original child points to original:", category["children"][0]["parent"] is category)
print("Shallow child points to shallow:", shallow_category["children"][0]["parent"] is shallow_category)
print("Shallow child still points to original:", shallow_category["children"][0]["parent"] is category)
print("Deep child points to deep:", deep_category["children"][0]["parent"] is deep_category)


Поверхнева копія дерева виглядає як копія тільки зверху. Вкладена дитина все ще посилається на старого батька, а `deepcopy()` перебудовує звʼязки всередині нової структури.


## Зрізи та копії списків

Зріз списку створює новий список, але елементи всередині не копіюються глибоко. Це ще один приклад поверхневої копії.


In [ ]:
initial = [1000, 2000, 3000, 4000]
backup = initial[:2]

print("backup:", backup)
print("Перший елемент спільний:", initial[0] is backup[0])
print("Списки різні:", initial is backup)


Для чисел це зазвичай безпечно, бо вони незмінні. Але для вкладених списків або словників така копія може несподівано змінити оригінал.


In [ ]:
initial = [[1000], [2000], [3000]]
backup = list(initial)

backup[0].append(4000)

print("Original:", initial)
print("Backup  :", backup)
print("Спільний перший вкладений список:", initial[0] is backup[0])


`list(initial)` створив новий зовнішній список, але вкладені списки залишилися тими самими. Тому зміна `backup[0]` відразу видна і в `initial`.


In [ ]:
values = [1000, 2000, 3000]
copy_with_list = list(values)
copy_with_unpacking = [*values]
copy_with_method = values.copy()

print(copy_with_list == values, copy_with_list is values)
print(copy_with_unpacking == values, copy_with_unpacking is values)
print(copy_with_method == values, copy_with_method is values)


Усі три способи створюють новий список верхнього рівня. Це зручно, але важливо памʼятати: це не глибоке копіювання елементів.


## Приховані копії в операціях

Деякі звичні операції створюють нові контейнери без явного слова `copy`. Найчастіше це видно у `sorted()` і конкатенації списків.


In [ ]:
values = [3000, 1000, 2000]
sorted_values = sorted(values)

print("Original after sorted():", values)
print("New sorted list      :", sorted_values)
print("Новий список:", sorted_values is not values)

values.sort()
print("Original after .sort():", values)


`sorted()` повертає новий список, а `.sort()` змінює існуючий. Але “на місці” не завжди означає “без жодної памʼяті”: алгоритм сортування може мати тимчасові буфери.


In [ ]:
records = [
    {"id": 3, "score": 91},
    {"id": 1, "score": 88},
    {"id": 2, "score": 95},
]

sorted_records = sorted(records, key=lambda item: item["score"])
print("Новий список після sorted:", sorted_records)
print("Оригінал після sorted    :", records)

records.sort(key=lambda item: item["score"])
print("Оригінал після .sort     :", records)


Для великих колекцій різниця між “новий список” і “змінити існуючий” може бути суттєвою. Це питання не тільки стилю, а й памʼяті.


In [ ]:
import time
import tracemalloc

chunks = [list(range(1000)) for _ in range(300)]


def measure_plus() -> None:
    result = []

    tracemalloc.start()
    start = time.perf_counter()

    for chunk in chunks:
        result = result + chunk

    elapsed = time.perf_counter() - start
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    print("Using +")
    print("Length:", len(result))
    print(f"Time: {elapsed:.4f} s")
    print(f"Peak memory: {peak:,} B")


def measure_extend() -> None:
    result = []

    tracemalloc.start()
    start = time.perf_counter()

    for chunk in chunks:
        result.extend(chunk)

    elapsed = time.perf_counter() - start
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    print("Using extend()")
    print("Length:", len(result))
    print(f"Time: {elapsed:.4f} s")
    print(f"Peak memory: {peak:,} B")


measure_plus()
print()
measure_extend()


`result = result + chunk` щоразу створює новий список і копіює старі посилання. `extend()` дописує елементи в існуючий список, тому зазвичай швидший і економніший. Варто звернути увагу, що навідміну від `result = result + chunk` варіант запису `result += chunk` буде економним, бо під капотом викличе `__iadd__`, а не `__add__`.


## Фрагментація: прихована деградація програми



Фрагментація тут означає не “памʼяті мало”, а “вільна памʼять розкидана дрібними шматками”. Python може звільнити окремі блоки, але arena не повернеться операційній системі, доки в ній лишається хоча б частина живих обʼєктів.

Тому в цьому експерименті важливо не лише **скільки** словників залишилось живими, а й **як вони розміщені**.


In [ ]:
iter_num = 2_000_000

initial = []
for i in range(iter_num):
    initial.append(None)

# створюємо 2 млн порожніх словників
for i in range(iter_num):
    initial[i] = {}

# Будемо дивитися почергово на 4 варіантах:

sliced = []                               # [1] без додаткових посилань
# sliced = initial[::2]                     # [2] кожен другий
# sliced = initial[iter_num // 2:]          # [3] друга половина
# sliced = initial[::100]                   # [4] кожен сотий

# видаляємо посилання з initial (sliced зберігає частину об’єктів живими)
for i in range(iter_num):
    initial[i] = None

# знову виділяємо словники для initial
for i in range(iter_num):
   initial[i] = {}


Почергово розкоментуйте кожну з чотирьох варіацій sliced і для кожної запустіть команду `export PYTHONMALLOCSTATS="True" && python3 DIR_TO_THIS_SCRIPT 2>result_{NUMBER_OF_RUN}.txt`

Етап побудови графіка. Тут читаються `result_1.txt` ... `result_4.txt`, створені на минулому етапі.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt


EXPERIMENT_DIR = Path.cwd()


def parse_malloc_stats(filename: Path) -> list[float]:
    values_mb: list[float] = []

    if not filename.exists():
        return values_mb

    with filename.open("r") as file:
        for line in file:
            if line.startswith("Total"):
                value_mb = float(line.split()[-1].replace(",", "")) / 1024 / 1024
                values_mb.append(value_mb)

    return values_mb


result_series = {
    "No elements": parse_malloc_stats(EXPERIMENT_DIR / "result_1.txt"),
    "Every second": parse_malloc_stats(EXPERIMENT_DIR / "result_2.txt"),
    "Second half": parse_malloc_stats(EXPERIMENT_DIR / "result_3.txt"),
    "Every hundredth": parse_malloc_stats(EXPERIMENT_DIR / "result_4.txt"),
}

for title, values in result_series.items():
    print(f"{title:16} -> {len(values)} точок")

if any(result_series.values()):
    plt.figure(figsize=(14, 10))

    for index, (title, values) in enumerate(result_series.items(), start=1):
        plt.subplot(2, 2, index)
        plt.title(title, loc="left")
        plt.plot(values)
        plt.grid(True, which="major", color="#666666", linestyle="-.")
        plt.minorticks_on()
        plt.grid(True, which="minor", color="#999999", linestyle="-", alpha=0.2)
        plt.ylabel("MB")

    plt.tight_layout()
    plt.show()
else:
    print("Даних для графіка ще немає. Виконайте дії, описані в попередній клітинці ноутбуку")

Як інтерпретувати
- `No elements`: старі словники майже не утримуються, тому allocator має найбільше шансів повторно використати або звільнити памʼять.
- `Every second`: половина словників лишається живою через один, тому багато arena залишаються “прошитими” живими обʼєктами.
- `Second half`: живих обʼєктів теж багато, але вони компактніші, тому частина арен може спорожніти помітніше.
- `Every hundredth`: живих обʼєктів мало, але вони розкидані, тому навіть 1% посилань може заважати повному звільненню багатьох арен.

Головна ідея: фрагментація залежить не тільки від кількості живих обʼєктів, а й від їхнього розташування в памʼяті.


## Генератори замість повних списків

Генератор не зберігає всі значення одразу. Він памʼятає стан обчислення і віддає наступний елемент тільки тоді, коли його попросили.


In [ ]:
import sys

numbers_list = [number for number in range(1_000_000)]
numbers_generator = (number for number in range(1_000_000))

print("Generator size:", sys.getsizeof(numbers_generator), "B")
print("List size     :", sys.getsizeof(numbers_list), "B")


`getsizeof()` не рахує всі вкладені числа у списку, але різниця вже помітна. Генератор корисний, коли нам не потрібні всі значення одночасно.


In [ ]:
import tracemalloc
from collections.abc import Generator


def read_log_file() -> list[str]:
    return [
        f"{index},ERROR,Disk full" if index % 100 == 0 else f"{index},INFO,Worker started"
        for index in range(100_000)
    ]


def collect_errors_list(lines: list[str]) -> list[str]:
    return [line for line in lines if "ERROR" in line]


def collect_errors_generator(lines: list[str]) -> Generator[str, None, None]:
    for line in lines:
        if "ERROR" in line:
            yield line


lines = read_log_file()

tracemalloc.start()
errors_list = collect_errors_list(lines)
_, peak_list = tracemalloc.get_traced_memory()
tracemalloc.stop()

tracemalloc.start()
errors_generator = collect_errors_generator(lines)
first_five_errors = [next(errors_generator) for _ in range(5)]
_, peak_generator = tracemalloc.get_traced_memory()
tracemalloc.stop()

print("Errors collected with list:", len(errors_list))
print("First five from generator:", first_five_errors)
print(f"Peak memory for list: {peak_list:,} B")
print(f"Peak memory for generator: {peak_generator:,} B")


Список збирає всі помилки в памʼяті. Генератор дозволяє обробляти потік поступово, тому добре підходить для логів, файлів і великих наборів даних.


## Слабкі посилання

`weakref` дає посилання на обʼєкт, але не утримує його живим. Це корисно для кешів, графів і систем підписок.


In [ ]:
import weakref


class Mountain:
    def __init__(self, name: str) -> None:
        self.name = name

    def __repr__(self) -> str:
        return f"Mountain({self.name})"


hoverla = Mountain("Hoverla")
mountain_ref = weakref.ref(hoverla)

print("Weak reference object:", mountain_ref)
print("Object through weakref:", mountain_ref())
print("Name:", mountain_ref().name)

del hoverla
print("After deleting strong reference:", mountain_ref())


Поки існує звичайне сильне посилання `hoverla`, обʼєкт доступний. Коли сильне посилання зникає, слабке посилання повертає `None`.


In [ ]:
import gc
import weakref


class UserProfile:
    def __init__(self, user_id: int) -> None:
        self.user_id = user_id

    def __repr__(self) -> str:
        return f"UserProfile({self.user_id})"


cache = weakref.WeakValueDictionary()

profile = UserProfile(42)
cache["user-42"] = profile

print("Before deleting strong reference:", dict(cache))

del profile
gc.collect()

print("After deleting strong reference:", dict(cache))


`WeakValueDictionary` не перетворює кеш на витік памʼяті. Якщо обʼєкт більше ніде не потрібен, запис у такому кеші зникає автоматично.


In [ ]:
import gc
import weakref


class Parent:
    def __init__(self, name: str) -> None:
        self.name = name
        self.children: list["Child"] = []

    def __repr__(self) -> str:
        return f"Parent({self.name})"


class Child:
    def __init__(self, name: str, parent: Parent) -> None:
        self.name = name
        self.parent = weakref.ref(parent)

    def __repr__(self) -> str:
        return f"Child({self.name})"


parent = Parent("Python")
child = Child("Memory", parent)
parent.children.append(child)

print("Child sees parent:", child.parent())

del parent
gc.collect()

print("Child sees parent after parent deletion:", child.parent())


Посилання “дитина → батько” часто створює цикл. Слабке посилання дозволяє дитині знайти батька, але не заважає звільнити його з памʼяті.


In [ ]:
import gc
import weakref


class Listener:
    def __init__(self, name: str) -> None:
        self.name = name

    def handle(self, message: str) -> None:
        print(f"{self.name} received: {message}")

    def __repr__(self) -> str:
        return f"Listener({self.name})"


class EventBus:
    def __init__(self) -> None:
        self.listeners = weakref.WeakSet()

    def subscribe(self, listener: Listener) -> None:
        self.listeners.add(listener)

    def notify(self, message: str) -> None:
        for listener in self.listeners:
            listener.handle(message)


bus = EventBus()
listener = Listener("Dashboard")
bus.subscribe(listener)

print("Before deleting listener:", list(bus.listeners))
bus.notify("New alert")

del listener
gc.collect()

print("After deleting listener:", list(bus.listeners))
bus.notify("Another alert")


У системах подій слабкі посилання допомагають не тримати слухачів у памʼяті випадково. Якщо обʼєкт зник, `WeakSet` прибере його сам.


## `__slots__`: менше службової памʼяті

Звичайний Python-обʼєкт має `__dict__` для динамічних атрибутів. `__slots__` задає фіксований набір полів і прибирає цей словник.


In [ ]:
class CityRegular:
    def __init__(self, name: str, population: int) -> None:
        self.name = name
        self.population = population


class CitySlotted:
    __slots__ = ("name", "population")

    def __init__(self, name: str, population: int) -> None:
        self.name = name
        self.population = population


regular_city = CityRegular("Kharkiv", 1_300_000)
slotted_city = CitySlotted("Kharkiv", 1_300_000)

print("Regular __dict__:", regular_city.__dict__)
print("Slotted has __dict__:", hasattr(slotted_city, "__dict__"))


`__slots__` добре підходить для багатьох однотипних обʼєктів. Ціна: менше гнучкості, бо не можна просто додати будь-який новий атрибут.


In [ ]:
import sys
from dataclasses import dataclass


@dataclass
class GeoPointRegular:
    lat: float
    lon: float
    city: str


@dataclass(slots=True)
class GeoPointSlotted:
    lat: float
    lon: float
    city: str


regular_points = [
    GeoPointRegular(50.00, 36.22, "Kharkiv")
    for _ in range(100_000)
]

slotted_points = [
    GeoPointSlotted(50.00, 36.22, "Kharkiv")
    for _ in range(100_000)
]

regular_one = regular_points[0]
slotted_one = slotted_points[0]

print("Regular has __dict__:", hasattr(regular_one, "__dict__"))
print("Slotted has __dict__:", hasattr(slotted_one, "__dict__"))

print("One regular object:", sys.getsizeof(regular_one))
print("One slotted object:", sys.getsizeof(slotted_one))

regular_total = sum(sys.getsizeof(point) + sys.getsizeof(point.__dict__) for point in regular_points)
slotted_total = sum(sys.getsizeof(point) for point in slotted_points)

print("Approx total for regular:", regular_total, "B")
print("Approx total for slotted:", slotted_total, "B")
print("Approx saved:", regular_total - slotted_total, "B")


Один обʼєкт може здаватися дрібницею, але на сотнях тисяч записів різниця стає помітною. Саме тому структура даних важлива для памʼяті.


## Словники, `namedtuple` і dataclass зі слотами

Список словників дуже гнучкий, але має великі накладні витрати. Якщо форма запису фіксована, компактні структури можуть суттєво зменшити памʼять.


In [ ]:
from pympler import asizeof
from collections import namedtuple
from dataclasses import dataclass

N = 1_000_000

PersonTuple = namedtuple("PersonTuple", ["id", "name", "age"])

@dataclass(slots=True)
class PersonDataClass:
    id: int
    name: str
    age: int


data_dict = [{"id": i, "name": f"name{i}", "age": i % 100} for i in range(N)]
data_tuple = [PersonTuple(i, f"name{i}", i % 100) for i in range(N)]
data_dataclass = [PersonDataClass(i, f"name{i}", i % 100) for i in range(N)]


print("List[dict]:", round(asizeof.asizeof(data_dict) / 1024**2, 2), "MB")
print("List[namedtuple]:", round(asizeof.asizeof(data_tuple) / 1024**2, 2), "MB")
print("List[dataclass(slots=True)]:", round(asizeof.asizeof(data_dataclass) / 1024**2, 2), "MB")

Це порівняння показує головну думку: однакові дані можна зберігати дуже по-різному. Якщо полів небагато і схема стабільна, `namedtuple` або `dataclass(slots=True)` часто кращі за словники.


## Підсумок

Копії в Python часто приховані: зрізи, `sorted()`, `+`, `list()` і розпакування створюють нові контейнери. Для великих даних варто свідомо обирати між `copy()`, `deepcopy()`, генераторами, слабкими посиланнями та компактними структурами на кшталт `__slots__`.
